# AUROC: does the predicted |error| find where the model errs?

Easy vs hard residues from the observed |error| (`oof_resid`), scored with the predicted |error| (`pred`).

In [ ]:
import os
os.makedirs("Figs", exist_ok=True)  # figures are written to ./Figs

import json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score

def set_style(base=14, family="Arial"):
    plt.rcParams.update({"font.family": family, "font.size": base,
        "axes.titlesize": base+2, "axes.labelsize": base, "xtick.labelsize": base-2,
        "ytick.labelsize": base-2, "legend.fontsize": base-2, "svg.fonttype": "none"})
set_style()

FIG5 = "../../data/analysis/Fig5"
AF   = "../../data/analysis/Fig4/all_features"
FIG  = "Figs"

# ---- CONFIG ----
PRED_PATH = f"{FIG5}/predictions_N_ensemble_10seed_sorted_centered_err.json"

def load_oof_err(pred_path):
    """Per-residue df: entry, resnum, pred (predicted |error|, ppm), acerr (OOF ground-truth
    |error|, already offset-corrected). Valid residues = mask & oof_mask (real residue AND
    OOF ground-truth coverage) — same contract as analyze_oof_err_predictions_sorted_centered.ipynb.
    No 4sigma-BMRB gate / per-protein offset correction here: oof_resid already has that folded in."""
    raw = json.load(open(pred_path))
    pred_d, mask_d, oof_resid_d, oof_mask_d = raw["pred"], raw["mask"], raw["oof_resid"], raw["oof_mask"]
    rows = []
    length_mismatch = 0
    for e in pred_d:
        p = np.asarray(pred_d[e], dtype=np.float64)
        m = np.asarray(mask_d[e], dtype=bool)
        g = np.asarray(oof_resid_d[e], dtype=np.float64)
        om = np.asarray(oof_mask_d[e], dtype=bool)
        n = min(len(p), len(m), len(g), len(om))
        if len({len(p), len(m), len(g), len(om)}) > 1:
            length_mismatch += 1
        valid = m[:n] & om[:n]
        idx = np.where(valid)[0]
        pv, gv = p[:n][valid], g[:n][valid]
        finite = np.isfinite(pv) & np.isfinite(gv)
        for i, pp, gg in zip(idx[finite], pv[finite], gv[finite]):
            rows.append((e, int(i) + 1, float(pp), float(gg)))
    df = pd.DataFrame(rows, columns=["entry", "resnum", "pred", "acerr"])
    if length_mismatch:
        print(f"WARNING: {length_mismatch} entries had mismatched pred/mask/oof_resid/oof_mask "
              f"lengths -- truncated to the shortest.")
    return df

d = load_oof_err(PRED_PATH)
print(f"OOF |error| predictions | rows={len(d)}  proteins={d.entry.nunique()}")
print(f"pred range {d.pred.min():.2f}-{d.pred.max():.2f} ppm | acerr range {d.acerr.min():.2f}-{d.acerr.max():.2f} ppm")
print(f"Spearman(pred, acerr) = {spearmanr(d.pred, d.acerr).correlation:.3f}")

In [ ]:
# ---- pred vs structural features vs both (SAME OOF acerr target, matched residues) ----
# Structural features sourced from all_features/per_residue_records_full.json (Grouping_geom.ipynb's
# merge_features.py + add_plddt_ph_temp.py output: sasa base + tdi/tdn/p_exchange + Talos-N s2 +
# AF2 geometry + ph/temperature/plddt), same source and column set as Grouping_geom.ipynb.
GEOM_CAT = ["prev_aa", "next_aa", "hb_HN_exists", "is_prepro"]
GEOM_NUM = ["phi_sin","phi_cos","psi_sin","psi_cos","psi_prev_sin","psi_prev_cos",
            "phi_next_sin","phi_next_cos","chi1_sin","chi1_cos",
            "hb_HN_dist","hb_HN_cosA","hb_HN_cosH", "hb_HN_energy","rc_N","pack_N8"]

prr = json.load(open(f"{AF}/per_residue_records_full.json"))
brows = [(e, r["resnum"], r["residue_type"], r["tdi"], r["tdn"], r["ss_class"], r["exposure"],
          r["s2"], r["ph"], r["temperature"], r["p_exchange"], r["plddt"],
          *[r[c] for c in GEOM_CAT], *[r[c] for c in GEOM_NUM])
         for e, rl in prr.items() for r in rl]
base = pd.DataFrame(brows, columns=["entry","resnum","aa","tdi","tdn","ss","exposure",
                                    "s2","ph","temperature","p_exchange","plddt"] + GEOM_CAT + GEOM_NUM)

BASE_CAT = ["aa","tdi","tdn","ss","exposure"]; BASE_NUM = ["s2","ph","temperature","p_exchange", "plddt"]
# drop residues missing a base feature (s2 is None for TALOS-uncovered N-term residues)
base = base.dropna(subset=BASE_CAT + BASE_NUM).reset_index(drop=True)

M = d.merge(base, on=["entry","resnum"], how="inner")
for c in BASE_NUM+GEOM_NUM:
    M[c] = M[c].fillna(M[c].median())
for c in GEOM_CAT:
    M[c] = M[c].fillna("NA")

# easy/hard on the matched subset (so all feature sets use identical residues/labels)
yy = M["acerr"].values
lo2, hi2 = np.quantile(yy, 0.50), np.quantile(yy, 0.90)
M["cls"] = np.where(yy <= lo2, 0, np.where(yy >= hi2, 1, -1))
sub = M[M.cls >= 0].copy()
Ym = sub["cls"].values; gm = sub["entry"].values
print(f"matched residues: {len(M)} | easy/hard subset {len(sub)} | proteins {sub.entry.nunique()}")

def evaluate(name, cat, num):
    trs = []
    if cat: trs.append(("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat))
    if num: trs.append(("num", StandardScaler(), num))
    pipe = Pipeline([("pre", ColumnTransformer(trs)),
                     ("m", HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05,
                          max_iter=400, l2_regularization=1.0, class_weight="balanced", random_state=0))])
    pr = cross_val_predict(pipe, sub[cat+num], Ym, cv=GroupKFold(5), groups=gm,
                           method="predict_proba")[:,1]
    a = roc_auc_score(Ym, pr); ap = average_precision_score(Ym, pr)
    print(f"  {name:24s} ROC-AUC={a:.3f}  AP={ap:.3f}")
    return a

STR_CAT = BASE_CAT + GEOM_CAT; STR_NUM = BASE_NUM + GEOM_NUM

In [ ]:
# ===== Panel A with protein-cluster-bootstrap 95% CI + paired test (structure+pred vs pred) =====
def oof_proba(cat, num):
    trs = []
    if cat: trs.append(("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat))
    if num: trs.append(("num", StandardScaler(), num))
    pipe = Pipeline([("pre", ColumnTransformer(trs)),
                     ("m", HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05,
                          max_iter=400, l2_regularization=1.0, class_weight="balanced", random_state=0))])
    return cross_val_predict(pipe, sub[cat+num], Ym, cv=GroupKFold(5), groups=gm,
                             method="predict_proba")[:, 1]

uniq = np.unique(gm); pidx = {p: np.where(gm == p)[0] for p in uniq}
rng = np.random.default_rng(0)

ci = lambda v: np.percentile(v, [2.5, 97.5])


In [ ]:
# ===== Combined: baseline vs pred-only vs structure-only vs structure+pred, one bootstrap, one figure =====
BASE_MIN_CAT = ["aa", "ss", "exposure"]

p_base = oof_proba(BASE_MIN_CAT, [])
p_pred = oof_proba([], ["pred"])
p_str  = oof_proba(STR_CAT, STR_NUM)
p_both = oof_proba(STR_CAT, STR_NUM + ["pred"])

In [ ]:
probs = [p_base, p_str, p_pred, p_both]
labels4 = ["baseline\n(aa, ss, exposure)", "all structure\nfeatures only", "model\nprediction", 
           "structure features\n+ model prediction"]

A4 = {i: [] for i in range(4)}
for _ in range(2000):
    idx = np.concatenate([pidx[p] for p in rng.choice(uniq, uniq.size, replace=True)])
    yv = Ym[idx]
    if len(np.unique(yv)) < 2:
        continue
    for i, pr in enumerate(probs):
        A4[i].append(roc_auc_score(yv, pr[idx]))

means4 = [np.mean(A4[i]) for i in range(4)]
print("=== baseline vs pred-only vs structure-only vs structure+pred (cluster-bootstrap 95% CI, 2000x) ===")
for nm, i in zip(labels4, range(4)):
    lo, hi = ci(A4[i]); print(f"  {nm:28s} {means4[i]:.3f}  [{lo:.3f}, {hi:.3f}]")

In [ ]:
# ---- plot ----
def set_style(base=14, family="Arial"):
    plt.rcParams.update({"font.family": family, "font.size": base,
        "axes.titlesize": base+2, "axes.labelsize": base, "xtick.labelsize": base - 1,
        "ytick.labelsize": base, "legend.fontsize": base-2, "svg.fonttype": "none"})
    
labels4 = ["aa, ss, exposure", "all\ndescriptors", "prediction", 
           "prediction\n+\nall descriptors"]

set_style(19)
errs4 = [[means4[i] - ci(A4[i])[0] for i in range(4)], [ci(A4[i])[1] - means4[i] for i in range(4)]]
fig, ax = plt.subplots(figsize=(7, 7))
bars = ax.bar(labels4, means4, yerr=errs4, capsize=5,
              color=["#999999", "#61AFEF", "#56B6C2", "#3F729B"], edgecolor="white",
              error_kw=dict(elinewidth=1.4, ecolor="#444"))
for b, v, i in zip(bars, means4, range(4)):
    ax.text(b.get_x()+b.get_width()/2, ci(A4[i])[1]+0.006, f"{v:.3f}", ha="center", va="bottom", fontsize=19, family="Arial")
ax.set_ylim(0.5, max(ci(A4[i])[1] for i in range(4))+0.03)
ax.set_ylabel("AUROC (easy vs hard)", fontsize=19, family="Arial")
# ax.grid(axis="y", alpha=0.3, ls="--")
for i, tick_label in enumerate(ax.get_xticklabels()):
    if i >= len(labels4) - 2:
        tick_label.set_fontweight("bold")
plt.tight_layout()
plt.savefig(f"{FIG}/T_prederr_baseline_vs_all_conditions_auroc_ci.svg", bbox_inches="tight")
plt.savefig(f"{FIG}/T_prederr_baseline_vs_all_conditions_auroc_ci.png", dpi=300, bbox_inches="tight")
plt.show()